# Agent 365 Lander (optional source)

Lands the **Agents 365** registry export (the MAC `Agents_*.csv`) into the Lakehouse Delta table
`dbo.agents_365`, which the Fabric dashboard reads via `FabricTable("agents_365")`.

Keeping Agent 365 in the Lakehouse (rather than a SharePoint URL in the report) keeps the Fabric
model **100% Lakehouse-sourced** — no gateway, no privacy-firewall, Direct Lake eligible.

**How to use:** drop the Agents 365 CSV at `Files/agent365/agents.csv` (or point `SOURCE_CSV` at a
OneLake shortcut), attach this notebook to the `CopilotAdoptionLake` Lakehouse, and Run all.
If the file is absent the dashboard's Agents 365 table simply loads empty (it is an optional source).

In [ ]:
# === CONFIG ===
SOURCE_CSV   = 'Files/agent365/agents.csv'   # drop the Agents 365 (MAC) export here, or a OneLake shortcut path
OUTPUT_TABLE = 'dbo.agents_365'              # Delta table read by the dashboard's Agents 365 table
WRITE_MODE   = 'overwrite'                   # 'overwrite' for full snapshots; 'append' for incremental

In [ ]:
# === LAND CSV -> Delta (column mapping preserves spaced header names like 'Agent name') ===
import notebookutils

def _exists(path):
    try:
        d = '/'.join(path.split('/')[:-1])
        return any(fi.name == path.split('/')[-1] for fi in notebookutils.fs.ls(d))
    except Exception:
        return False

if not _exists(SOURCE_CSV):
    print(f"Agent 365 export not found at {SOURCE_CSV} \u2014 nothing landed. "
          f"The dashboard's Agents 365 table will load empty (optional source).")
else:
    df = (spark.read
          .option('header', True)
          .option('multiLine', True)
          .option('escape', '"')
          .option('encoding', 'UTF-8')
          .csv(SOURCE_CSV))
    print(f'agents_365 rows: {df.count():,} | columns: {len(df.columns)}')
    (df.write.mode(WRITE_MODE)
       .option('overwriteSchema', 'true')
       .option('delta.columnMapping.mode', 'name')
       .format('delta').saveAsTable(OUTPUT_TABLE))
    print(f'wrote -> {OUTPUT_TABLE} ({WRITE_MODE})')